In [3]:
import os

os.makedirs("data", exist_ok=True)
os.makedirs("model_building", exist_ok=True)
os.makedirs("deployment", exist_ok=True)
os.makedirs("hosting", exist_ok=True)
os.makedirs(".github/workflows", exist_ok=True)
print("Folder structure created successfully.")

Folder structure created successfully.


In [5]:
%%writefile model_building/data_register.py
import os
import pandas as pd
from datasets import Dataset
from huggingface_hub import HfApi, create_repo

# *** UPDATE THIS TO YOUR HF USERNAME ***
repo_id = "your-username/tourism-package-data"
repo_type = "dataset"

api = HfApi(token=os.getenv("HF_TOKEN"))

# Check if space exists, create if not
try:
    api.repo_info(repo_id=repo_id, repo_type=repo_type)
    print(f"Space '{repo_id}' already exists.")
except:
    create_repo(repo_id=repo_id, repo_type=repo_type, exist_ok=True)
    print(f"Created space '{repo_id}'.")

# Load local data and push to hub
df = pd.read_csv('Advanced-Machine-Learning-and-MLOps/data/tourism.csv')
hf_dataset = Dataset.from_pandas(df)
hf_dataset.push_to_hub(repo_id, token=os.getenv("HF_TOKEN"))

print(f"Data successfully registered at {repo_id}")

Writing model_building/data_register.py


In [6]:
%%writefile model_building/prep.py
import os
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

# *** UPDATE THIS TO YOUR HF USERNAME ***
repo_id = "your-username/tourism-package-data"

# Load dataset
dataset = load_dataset(repo_id, token=os.getenv("HF_TOKEN"))
df = dataset['train'].to_pandas()

# Clean Data
columns_to_drop = ['Unnamed: 0', 'CustomerID']
df_clean = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

for col in df_clean.select_dtypes(include=['float64', 'int64']).columns:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)
for col in df_clean.select_dtypes(include=['object']).columns:
    df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

df_encoded = pd.get_dummies(df_clean, drop_first=True)

# Split 
X = df_encoded.drop('ProdTaken', axis=1)
y = df_encoded['ProdTaken']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Upload splits back to Hub
Dataset.from_pandas(pd.concat([X_train, y_train], axis=1)).push_to_hub(repo_id + "-train", token=os.getenv("HF_TOKEN"))
Dataset.from_pandas(pd.concat([X_test, y_test], axis=1)).push_to_hub(repo_id + "-test", token=os.getenv("HF_TOKEN"))
print("Data Preparation and Splitting complete.")

Writing model_building/prep.py


In [7]:
%%writefile model_building/train.py
import os
import mlflow
import joblib
from datasets import load_dataset
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from huggingface_hub import HfApi, create_repo

# *** UPDATE THIS TO YOUR HF USERNAME ***
dataset_repo = "your-username/tourism-package-data"
model_repo = "your-username/tourism-prediction-model"

train_data = load_dataset(dataset_repo + "-train", token=os.getenv("HF_TOKEN"))['train'].to_pandas()
test_data = load_dataset(dataset_repo + "-test", token=os.getenv("HF_TOKEN"))['train'].to_pandas()

X_train, y_train = train_data.drop('ProdTaken', axis=1), train_data['ProdTaken']
X_test, y_test = test_data.drop('ProdTaken', axis=1), test_data['ProdTaken']

mlflow.set_experiment("Tourism_Package_Prediction")
best_acc, best_model = 0, None

for lr in [0.01, 0.1]:
    for depth in [3, 5]:
        with mlflow.start_run():
            model = XGBClassifier(learning_rate=lr, max_depth=depth, random_state=42)
            model.fit(X_train, y_train)
            acc = accuracy_score(y_test, model.predict(X_test))
            
            mlflow.log_param("learning_rate", lr)
            mlflow.log_param("max_depth", depth)
            mlflow.log_metric("accuracy", acc)
            
            if acc > best_acc:
                best_acc, best_model = acc, model

# Register Model in Hugging Face
model_filename = "xgboost_tourism_model.joblib"
joblib.dump(best_model, model_filename)
api = HfApi(token=os.getenv("HF_TOKEN"))

try:
    api.repo_info(repo_id=model_repo, repo_type="model")
except:
    create_repo(repo_id=model_repo, repo_type="model", exist_ok=True)

api.upload_file(
    path_or_fileobj=model_filename,
    path_in_repo=model_filename,
    repo_id=model_repo,
    repo_type="model"
)
print("Model trained, tracked, and pushed successfully.")

Writing model_building/train.py


In [8]:
%%writefile deployment/Dockerfile
FROM python:3.9
WORKDIR /app
COPY . .
RUN pip3 install -r requirements.txt
RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user \
	PATH=/home/user/.local/bin:$PATH
WORKDIR $HOME/app
COPY --chown=user . $HOME/app
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

Writing deployment/Dockerfile


In [9]:
%%writefile deployment/app.py
import streamlit as st
import joblib
from huggingface_hub import hf_hub_download

# *** UPDATE THIS TO YOUR HF USERNAME ***
repo_id = "your-username/tourism-prediction-model"
model_filename = "xgboost_tourism_model.joblib"

model_path = hf_hub_download(repo_id=repo_id, filename=model_filename)
model = joblib.load(model_path)

st.title("Wellness Tourism Package Prediction")
st.write("Predict the likelihood of a customer purchasing the package based on specific factors.")

age = st.number_input("Age", 18, 100, 30)
duration = st.number_input("Pitch Duration (mins)", 1.0, 50.0, 10.0)

if st.button("Predict"):
    st.info("In a full deployment, these inputs will be mapped to the trained model features to output a 0 or 1.")

Writing deployment/app.py


In [10]:
%%writefile deployment/requirements.txt
pandas==2.2.2
huggingface_hub==0.32.6
streamlit==1.43.2
joblib==1.5.1
scikit-learn==1.6.0
xgboost==2.1.4

Writing deployment/requirements.txt


In [11]:
%%writefile hosting/hosting.py
from huggingface_hub import HfApi
import os

api = HfApi(token=os.getenv("HF_TOKEN"))
# *** UPDATE THIS TO YOUR HF USERNAME AND TARGET SPACE NAME ***
SPACE_REPO = "your-username/Tourism-Prediction-Space" 

api.upload_folder(
    folder_path="Advanced-Machine-Learning-and-MLOps/deployment",     
    repo_id=SPACE_REPO,          
    repo_type="space",                      
    path_in_repo="",                          
)
print("Deployment folder pushed to Hugging Face Space.")

Writing hosting/hosting.py


In [12]:
%%writefile requirements.txt
huggingface_hub==0.32.6
datasets==3.6.0
pandas==2.2.2
scikit-learn==1.6.0
xgboost==2.1.4
mlflow==3.0.1
joblib==1.5.1

Writing requirements.txt


In [13]:
%%writefile .github/workflows/pipeline.yml
name: Tourism MLOps Pipeline

on:
  push:
    branches:
      - main 

jobs:
  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r Advanced-Machine-Learning-and-MLOps/requirements.txt
      - name: Upload Dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python Advanced-Machine-Learning-and-MLOps/model_building/data_register.py

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r Advanced-Machine-Learning-and-MLOps/requirements.txt
      - name: Run Data Preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python Advanced-Machine-Learning-and-MLOps/model_building/prep.py

  model-training:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r Advanced-Machine-Learning-and-MLOps/requirements.txt
      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &
          sleep 5
      - name: Model Building
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python Advanced-Machine-Learning-and-MLOps/model_building/train.py

  deploy-hosting:
    runs-on: ubuntu-latest
    needs: [model-training, data-prep, register-dataset]
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r Advanced-Machine-Learning-and-MLOps/requirements.txt
      - name: Push files to Frontend Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python Advanced-Machine-Learning-and-MLOps/hosting/hosting.py

Writing .github/workflows/pipeline.yml
